In [0]:
dbutils.library.restartPython()

In [0]:
%pip install /Workspace/Users/gpuchalski@kumc.edu/PFTSleep/

In [0]:
%pip freeze

In [0]:
# -------------------------
# 1) Imports
# -------------------------
from pathlib import Path
import os
import time
import numpy as np
import torch

from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from pftsleep.transformers import PatchTFTSimple
from pftsleep.slumber import (
    SelfSupervisedTimeFrequencyDataset,
    ALL_FREQUENCY_FILTERS,
    VOLTAGE_CHANNELS
)




In [0]:
# -------------------------
# 2) User-configurable variables
# -------------------------
# ZARR FILES
zarr_dir = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/zarrs")
num_files = 768

#hyp_dir = Path("/home/gpuchalski/projects/PFTSleep/hypnograms")
# SIGNAL / WINDOWING
frequency = 125
win_length = 750
hop_length = 750
max_seq_len_sec = 8 * 3600
window_size_sec = win_length / frequency

# PERFORMANCE
workers = 8
device = "cuda" if torch.cuda.is_available() else "cpu"

# ENCODERS / CHECKPOINTS (add more here as you expand)
pft_ckpt_path = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/models/pft_sleep_encoder.ckpt")

# OUTPUT DIRECTORIES
save_PCA_plot_here = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/pca_plots/")
save_UMAP_plot_here = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/umap_plots/")
cache_dir = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/cache_dir/")

save_PCA_plot_here.mkdir(parents=True, exist_ok=True)
save_UMAP_plot_here.mkdir(parents=True, exist_ok=True)
cache_dir.mkdir(parents=True, exist_ok=True)

# PCA
given_n_components = 512
pca_target_var = 0.95  # "PCA95% variance"

# UMAP (visualization only)
umap_n_neighbors = 15
umap_min_dist = 0.1
umap_n_components = 3
umap_metric = "euclidean"
umap_random_state = 42

# K-selection sweep
k_values = list(range(8, 16))  # adjust as needed
seeds = [0, 1, 2, 3, 4]

# LSH parameters (raw-space comparison)
lsh_n_bits = 16
lsh_top_buckets = 2000


In [0]:
# -------------------------
# 3) Channel definitions
# -------------------------
channels = [
    ["ECG", "ECG (L-R)", "EKG"],
    ["EOG(L)", "E1", "E1-M1", "EOG-L"],
    ["EMG", "cchin_1", "chin", "EMG (L-R)"],
    ["EEG", "C3-M2", "C4-M1", "C3-M2", "EEG3"],
    ["SaO2", "spo2", "SpO2"],
    ["THOR RES", "thorax", "Thoracic", "Chest", "Thor"],
    ["ABDO RES", "abdomen", "Abdominal", "ABD", "Abdo"],
]
c_in = len(channels)


In [0]:
# -------------------------
# 4) Load Zarrs
# -------------------------
zarr_files = sorted([p for p in zarr_dir.glob("*.zarr")])[:num_files]
print(f"Found {len(zarr_files)} zarr files (using num_files={num_files}).")
assert len(zarr_files) > 0, "No Zarr files found."


In [0]:

# -------------------------
# 5) Dataset / Loader
# -------------------------
max_seq_len = int(max_seq_len_sec * frequency)

dataset = SelfSupervisedTimeFrequencyDataset(
    zarr_files=zarr_files,
    channels=channels,
    frequency=int(frequency),
    trim_wake_epochs=True,
    return_hypnogram_every_sec=30,
    hypnogram_frequency=1,
    hypnogram_padding_mask=-100,
    scale_channels=False,
    start_offset_sec=0,
    clip_interpolations=None,
    include_partial_samples=True,
    return_sequence_padding_mask=True,
    butterworth_filters=ALL_FREQUENCY_FILTERS,
    median_filter_kernel_size=3,
    voltage_channels=VOLTAGE_CHANNELS,
    max_seq_len_sec=int(max_seq_len_sec),
    sample_seq_len_sec=int(max_seq_len_sec),
    sample_stride_sec=int(max_seq_len_sec),
)

loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    num_workers=8,
    persistent_workers=False,
    pin_memory=True,
)

print("Dataset + loader ready.")

In [0]:
# -------------------------
# 6) Encoder registry (extendable)
# -------------------------
def build_pftsleep_encoder() -> torch.nn.Module:
    model = PatchTFTSimple(
        c_in=c_in,
        win_length=win_length,
        hop_length=hop_length,
        max_seq_len=max_seq_len,
        use_revin=True,
        dim1reduce=False,
        affine=True,
        use_flash_attn=False,
        augmentations=["jitter_zero_mask"],
        mask_ratio=0.1,
        n_layers=3,
        d_model=512,
        n_heads=4,
        shared_embedding=False,
        d_ff=2048,
        norm="BatchNorm",
        attn_dropout=0.0,
        dropout=0.1,
        act="gelu",
        res_attention=True,
        pre_norm=False,
        store_attn=False,
        pretrain_head=False,
    )
    return model

def load_pftsleep_weights(model: torch.nn.Module, ckpt_path: Path) -> torch.nn.Module:
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = {
        k.replace("model.", ""): v
        for k, v in ckpt["state_dict"].items()
        if k.startswith("model.")
    }
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    return model

# Placeholder hooks for other encoders (SleepFM, etc.)
# Implement build/load functions similarly, then register below.
def build_sleepfm_encoder():
    raise NotImplementedError("Add SleepFM encoder construction here.")

def load_sleepfm_weights(model, ckpt_path: Path):
    raise NotImplementedError("Add SleepFM checkpoint loading here.")

ENCODER_REGISTRY = {
    "PFTSleep": {
        "build": build_pftsleep_encoder,
        "load": lambda m: load_pftsleep_weights(m, pft_ckpt_path),
    },
    # "SleepFM": {"build": build_sleepfm_encoder, "load": lambda m: load_sleepfm_weights(m, sleepfm_ckpt_path)},
}

In [0]:
# -------------------------
# 7) Extract + cache latents per encoder
# -------------------------
def cache_tag(
    encoder_name: str,
    zarr_dir: Path,
    num_files: int,
    frequency: int,
    win_length: int,
    hop_length: int,
    max_seq_len_sec: int,
) -> str:
    # Stable-ish cache key for your current experimental setup
    # (If you change anything above, it naturally creates a new cache file)
    return (
        f"{encoder_name}"
        f"__files{num_files}"
        f"__freq{frequency}"
        f"__win{win_length}"
        f"__hop{hop_length}"
        f"__max{max_seq_len_sec}"
    )

def extract_latents_cached(
    encoder_name: str,
    encoder: torch.nn.Module,
    loader: DataLoader,
    cache_dir: Path,
    window_size_sec: float,
    device: str = "cpu",
):
    tag = cache_tag(
        encoder_name=encoder_name,
        zarr_dir=zarr_dir,
        num_files=num_files,
        frequency=frequency,
        win_length=win_length,
        hop_length=hop_length,
        max_seq_len_sec=max_seq_len_sec,
    )

    Z_path = cache_dir / f"Z__{tag}.npy"
    night_id_path = cache_dir / f"night_id__{tag}.npy"
    time_idx_path = cache_dir / f"time_idx__{tag}.npy"
    zarr_file_idx_path = cache_dir / f"zarr_file_idx__{tag}.npy"

    if Z_path.exists() and night_id_path.exists() and time_idx_path.exists() and zarr_file_idx_path.exists():
        print(f"[{encoder_name}] Loading cached latents:")
        Z = np.load(Z_path)
        night_id = np.load(night_id_path)
        time_idx = np.load(time_idx_path)
        zarr_file_idx = np.load(zarr_file_idx_path)
        print(f"  Z: {Z.shape}, nights: {len(np.unique(night_id))}")
        return Z, night_id, time_idx, zarr_file_idx

    print(f"[{encoder_name}] No cache found. Extracting latents (one-time cost)...")
    encoder = encoder.to(device)
    encoder.eval()

    latent_list = []
    night_ids = []
    time_idxs = []
    zarr_file_idxs = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(loader, desc=f"Encoding ({encoder_name})", unit="file")):
            x = batch[0].to(device)                    # [1, C, T]
            sequence_padding_mask = batch[2].to(device) # [1, T]

            z_latent = encoder(x, sequence_padding_mask=sequence_padding_mask)
            z_latent = z_latent.squeeze(0)            # [?, ?, ?] depending on model
            z_latent = z_latent.permute(0, 2, 1)      # match your earlier reshaping
            z_latent = z_latent.reshape(-1, 512)      # [N_windows, 512]

            z_np = z_latent.detach().cpu().numpy()
            latent_list.append(z_np)

            n_windows = z_np.shape[0]
            night_ids.append(np.full(n_windows, batch_idx, dtype=int))
            time_idxs.append(np.arange(n_windows, dtype=float) * window_size_sec)
            zarr_file_idxs.append(np.full(n_windows, batch_idx, dtype=int))  # batch_idx corresponds to zarr_files order

    Z = np.vstack(latent_list)
    night_id = np.concatenate(night_ids)
    time_idx = np.concatenate(time_idxs)
    zarr_file_idx = np.concatenate(zarr_file_idxs)

    np.save(Z_path, Z)
    np.save(night_id_path, night_id)
    np.save(time_idx_path, time_idx)
    np.save(zarr_file_idx_path, zarr_file_idx)

    print(f"[{encoder_name}] Saved cache:")
    print(f"  {Z_path.name}  shape={Z.shape}")

    return Z, night_id, time_idx, zarr_file_idx

# Build + load + extract for all encoders you enable
LATENTS = {}
for enc_name, spec in ENCODER_REGISTRY.items():
    model = spec["build"]()
    model = spec["load"](model)
    Z, night_id, time_idx, zarr_file_idx = extract_latents_cached(
        encoder_name=enc_name,
        encoder=model,
        loader=loader,
        cache_dir=cache_dir,
        window_size_sec=window_size_sec,
        device=device,
    )
    LATENTS[enc_name] = {
        "Z": Z,
        "night_id": night_id,
        "time_idx": time_idx,
        "zarr_file_idx": zarr_file_idx,
        "zarr_files": [str(p) for p in zarr_files],  # keep mapping for reference
    }

# Use PFTSleep latents for the rest of this notebook (swap if you add more)
Z = LATENTS["PFTSleep"]["Z"]
night_id = LATENTS["PFTSleep"]["night_id"]
time_idx = LATENTS["PFTSleep"]["time_idx"]
zarr_file_idx = LATENTS["PFTSleep"]["zarr_file_idx"]
zarr_files_list = LATENTS["PFTSleep"]["zarr_files"]

print("Final latent matrix Z:", Z.shape)
# To get the zarr file path for each point: zarr_files_list[zarr_file_idx]